# B2-019-attention-transformers — Practice p12 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** transformer-residual-layernorm, position-wise-feed-forward, transformer-block

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

The required recurrence is pre-norm: normalize only the sublayer input, then add the sublayer result to its unnormalized residual source. Thus y=x+attention(norm1(x),allowed), followed by z=y+ffn(norm2(y)). A zero sublayer is an exact identity through its residual path.

In [ ]:
import torch
from torch import nn

SEED = 20260808
ATOL = 1e-10
RTOL = 1e-10
torch.manual_seed(SEED)

x = torch.arange(24, dtype=torch.float64).reshape(2, 3, 4) / 10.0
allowed = torch.tril(torch.ones(2, 3, 3, dtype=torch.bool))
ATTENTION_WEIGHT = torch.tensor(
    [[1.0, 0.0, 0.0, 0.0],
     [0.0, 0.5, 0.0, 0.0],
     [0.0, 0.0, -0.5, 0.0],
     [0.25, 0.0, 0.0, 0.75]], dtype=torch.float64
)
FFN_WEIGHT_1 = torch.arange(32, dtype=torch.float64).reshape(8, 4) / 50.0 - 0.3
FFN_BIAS_1 = torch.linspace(-0.2, 0.2, 8, dtype=torch.float64)
FFN_WEIGHT_2 = torch.arange(32, dtype=torch.float64).reshape(4, 8) / 80.0 - 0.15
FFN_BIAS_2 = torch.tensor([0.05, -0.05, 0.1, -0.1], dtype=torch.float64)

class SuppliedAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(4, 4, bias=False, dtype=torch.float64)
    def forward(self, normalized_x, allowed):
        if allowed.dtype != torch.bool or allowed.shape != (2, 3, 3):
            raise ValueError("allowed must be Boolean with shape (2,3,3)")
        return self.projection(normalized_x)

def initialize_nonzero(block):
    with torch.no_grad():
        block.attention.projection.weight.copy_(ATTENTION_WEIGHT)
        block.ffn[0].weight.copy_(FFN_WEIGHT_1)
        block.ffn[0].bias.copy_(FFN_BIAS_1)
        block.ffn[2].weight.copy_(FFN_WEIGHT_2)
        block.ffn[2].bias.copy_(FFN_BIAS_2)

class PreNormBlock(nn.Module):
    def __init__(self, attention, width=4):
        super().__init__()
        self.attention = attention
        self.norm1 = nn.LayerNorm(width, dtype=torch.float64)
        self.norm2 = nn.LayerNorm(width, dtype=torch.float64)
        self.ffn = nn.Sequential(
            nn.Linear(4, 8, dtype=torch.float64),
            nn.ReLU(),
            nn.Linear(8, 4, dtype=torch.float64),
        )
    def forward(self, x, allowed):
        y = x + self.attention(self.norm1(x), allowed)
        z = y + self.ffn(self.norm2(y))
        return z

block = PreNormBlock(SuppliedAttention())
with torch.no_grad():
    block.attention.projection.weight.zero_()
    block.ffn[0].weight.zero_()
    block.ffn[0].bias.zero_()
    block.ffn[2].weight.zero_()
    block.ffn[2].bias.zero_()
z = block(x, allowed)
initialize_nonzero(block)
nonzero_z = block(x, allowed)
direct_y = x + block.attention(block.norm1(x), allowed)
direct_z = direct_y + block.ffn(block.norm2(direct_y))

### Answer check

In [ ]:
assert z.shape == nonzero_z.shape == (2, 3, 4)
assert z.dtype == nonzero_z.dtype == torch.float64
assert torch.allclose(z, x, atol=ATOL, rtol=RTOL)
assert torch.allclose(nonzero_z, direct_z, atol=ATOL, rtol=RTOL)
assert not torch.allclose(nonzero_z, x, atol=ATOL, rtol=RTOL)